# CachePilot 真实 GPU 验收

本 notebook 只验证真实 GPU 推理，不使用 DeterministicChatBackend、fake engine 或模拟模型结果。

重要：项目和 vLLM `0.24.0` 的锁定基线是 Python `3.13.15`。vLLM `0.24.0` 的约束是 Python `<3.15`，所以 Python `3.15` 不能作为本项目的 GPU 运行环境；notebook 会在模型加载前检查单卡、显存、磁盘、CUDA/PyTorch 与固定模型版本。

验收范围：真实 CUDA/GPU、真实 Qwen 固定 revision、真实 TorchExecutor、真实 vLLM、真实 CachePilot Gateway、逐请求 Trace 和 Prometheus 指标。环境出口只写入白名单字段，不打印环境变量或凭据。

## 1. 挂载仓库并固定工作目录

验收使用 Google Drive 中的仓库目录。安装命令会将 pip 临时文件和缓存放在 `/content`，减少 Drive FUSE 短暂断连对依赖安装的影响。

In [ ]:
from google.colab import drive
import os
import pathlib

os.chdir('/content')
drive.mount('/content/drive', force_remount=True)
WORKDIR = pathlib.Path('/content/drive/MyDrive/CachePilot')
if not WORKDIR.is_dir():
    raise FileNotFoundError(f'CachePilot repository not found: {WORKDIR}')
os.chdir(WORKDIR)
print('工作目录:', WORKDIR)

## 2. 单卡环境出口与硬门禁

这一步在安装依赖和加载模型前执行。它输出 GPU、显存、驱动、CUDA、Python、PyTorch、磁盘和模型 revision；必须恰好一张 GPU，显存至少 8 GiB，磁盘至少 10 GiB，且模型/tokenizer revision 固定。条件不满足时直接失败，不把较小显存机器的结果当成同一模型容量结论。

In [ ]:
import json
import re
import sys

from benchmarks.gpu_environment import collect_environment, validate_environment

def redact_secrets(value):
    patterns = (
        (r'(?i)(hf_token|huggingface_token|api[_-]?key|access[_-]?token|password)(\s*[=:]\s*)[^\s,;]+', r'\1\2[REDACTED]'),
        (r'(?i)bearer\s+[A-Za-z0-9._-]+', 'Bearer [REDACTED]'),
    )
    for pattern, replacement in patterns:
        value = re.sub(pattern, replacement, value)
    return value

environment = collect_environment(WORKDIR)
environment_failures = validate_environment(environment)
print(redact_secrets(json.dumps(environment, ensure_ascii=False, sort_keys=True, indent=2)))
if environment_failures:
    raise RuntimeError('GPU environment rejected before dependency install/model load: ' + '; '.join(environment_failures))
print('GPU_PYTHON_PREFLIGHT=PASS')


## 3. 安装锁定的真实 GPU 依赖

安装失败直接停止；不再降级为 CPU 或 fake 实现。

In [ ]:
import os
import pathlib
import subprocess
import sys
import time

project_packages = [
    'fastapi==0.115.12',
    'httpx==0.28.1',
    'pydantic==2.11.4',
    'starlette==0.46.2',
    'uvicorn==0.34.2',
]
gpu_packages = [
    'torch==2.11.0',
    'transformers==5.5.3',
    'tokenizers==0.22.2',
    'safetensors==0.6.2',
    'huggingface_hub==1.5.0',
    'vllm==0.24.0',
]
pip_temp = pathlib.Path('/content/cachepilot-pip-tmp')
pip_cache = pathlib.Path('/content/cachepilot-pip-cache')
pip_temp.mkdir(parents=True, exist_ok=True)
pip_cache.mkdir(parents=True, exist_ok=True)
pip_environment = os.environ.copy()
pip_environment['TMPDIR'] = str(pip_temp)
pip_environment['PIP_CACHE_DIR'] = str(pip_cache)

def install_packages(packages, attempts=2):
    command = [
        sys.executable, '-m', 'pip', 'install', '--upgrade',
        '--disable-pip-version-check', '--break-system-packages',
        '--retries', '8', '--timeout', '120', *packages,
    ]
    for attempt in range(1, attempts + 1):
        result = subprocess.run(
            command,
            cwd='/content',
            env=pip_environment,
            text=True,
            capture_output=True,
        )
        if result.stdout:
            print(redact_secrets(result.stdout[-6000:]))
        if result.returncode == 0:
            return
        if result.stderr:
            print(redact_secrets(result.stderr[-12000:]))
        if attempt < attempts:
            print(f'pip install failed (attempt {attempt}/{attempts}); retrying in 5 seconds...')
            time.sleep(5)
    raise subprocess.CalledProcessError(result.returncode, command)

install_packages(project_packages)
install_packages(gpu_packages)
os.environ['PYTHONPATH'] = str(WORKDIR) + os.pathsep + os.environ.get('PYTHONPATH', '')
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
print('GPU_DEPENDENCIES_INSTALL=PASS')

In [ ]:
from importlib import metadata
import json
import torch
import transformers

assert torch.__version__.split('+')[0] == '2.11.0', torch.__version__
assert transformers.__version__ == '5.5.3', transformers.__version__
assert metadata.version('vllm') == '0.24.0'
assert torch.cuda.is_available(), 'CUDA is not available'
runtime_environment = collect_environment(WORKDIR)
runtime_failures = validate_environment(runtime_environment, require_torch=True)
print(redact_secrets(json.dumps(runtime_environment, ensure_ascii=False, sort_keys=True, indent=2)))
if runtime_failures:
    raise RuntimeError('GPU runtime rejected before model load: ' + '; '.join(runtime_failures))
print('Torch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('vLLM:', metadata.version('vllm'))
print('CUDA:', torch.version.cuda)
print('Device:', torch.cuda.get_device_name(0))
print('GPU_RUNTIME=PASS')


## 4. 真实 TorchExecutor + CachePilot Gateway

加载固定 revision 的 Qwen，不注入 fake model/tokenizer。请求必须经过真实 GPU 生成，再检查 Gateway Trace 和 Prometheus。若 Colab 中的源码副本尚未包含 TorchExecutor 的 prompt 计数方法，notebook 会使用同一个真实 tokenizer 的适配器，不使用估算 token 数。

In [5]:
from httpx import ASGITransport, AsyncClient
from cachepilot.config.baseline import load_model_baseline
from cachepilot.executors import TorchExecutor, TorchExecutorConfig
from cachepilot.gateway.api import GatewaySettings, create_app

baseline_obj = load_model_baseline(WORKDIR / 'config/model.json')
torch_executor = TorchExecutor(
    TorchExecutorConfig(
        model_id=baseline_obj.model_id,
        tokenizer_id=baseline_obj.tokenizer_id,
        model_revision=baseline_obj.model_revision,
        tokenizer_revision=baseline_obj.tokenizer_revision,
        device='cuda',
        dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'float16',
        context_limit=baseline_obj.service_context_limit,
        trust_remote_code=baseline_obj.trust_remote_code,
    )
)
executor_counter = getattr(torch_executor, 'count_prompt_tokens', None)
if callable(executor_counter):
    real_token_counter = torch_executor
    print('TORCH_TOKEN_COUNTER=EXECUTOR')
else:
    class RealTokenizerPromptCounter:
        def __init__(self, tokenizer):
            self.tokenizer = tokenizer

        def count_prompt_tokens(self, request):
            messages = [
                {'role': message.role, 'content': message.content}
                for message in request.messages
            ]
            token_ids = self.tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True
            )
            shape = getattr(token_ids, 'shape', None)
            if shape is not None and len(shape) >= 2:
                return int(shape[-1])
            if isinstance(token_ids, (list, tuple)) and token_ids and isinstance(token_ids[0], (list, tuple)):
                return len(token_ids[0])
            return len(token_ids)

    real_token_counter = RealTokenizerPromptCounter(torch_executor.tokenizer)
    print('TORCH_TOKEN_COUNTER=REAL_TOKENIZER_ADAPTER')
torch_app = create_app(
    GatewaySettings(model_id=baseline_obj.model_id),
    backend=torch_executor,
    token_counter=real_token_counter,
)

async def run_real_torch_gateway():
    async with AsyncClient(
        transport=ASGITransport(app=torch_app),
        base_url='http://cachepilot',
    ) as client:
        ready = await client.get('/readyz')
        assert ready.status_code == 200 and ready.json()['status'] == 'ready'
        response = await client.post(
            '/v1/chat/completions',
            headers={'X-Tenant-ID': 'team-a', 'X-Request-ID': 'real-torch-gateway'},
            json={
                'model': baseline_obj.model_id,
                'messages': [{'role': 'user', 'content': 'Reply with a short greeting.'}],
                'stream': False,
                'max_tokens': 16,
            },
        )
        assert response.status_code == 200, response.text
        payload = response.json()
        output = payload['choices'][0]['message']['content']
        assert output.strip(), 'TorchExecutor returned empty output'
        trace_response = await client.get(
            '/v1/requests/real-torch-gateway',
            headers={'X-Tenant-ID': 'team-a'},
        )
        assert trace_response.status_code == 200
        trace = trace_response.json()['telemetry']
        assert trace['terminal_state'] == 'FINISHED'
        assert trace['prompt_tokens'] > 0
        assert trace['completion_tokens'] > 0
        assert trace['ttft_ms'] is not None
        metrics = (await client.get('/metrics')).text
        assert 'cachepilot_requests_total' in metrics
        assert 'cachepilot_ttft_seconds' in metrics
        for forbidden in ('request_id=', 'prompt=', 'prefix_key='):
            assert forbidden not in metrics
        return output, trace

torch_output, torch_trace = await run_real_torch_gateway()
print('TORCH_OUTPUT:', torch_output)
print('TORCH_TRACE:', torch_trace)
print('REAL_TORCH_GATEWAY=PASS')
print('REAL_TORCH_PROMETHEUS=PASS')

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

TORCH_TOKEN_COUNTER=EXECUTOR


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


TORCH_OUTPUT: Hello! How can I assist you today?
TORCH_TRACE: {'request_id': 'real-torch-gateway', 'tenant_id': 'team-a', 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'prompt_tokens': 35, 'admission_status': 'ADMITTED', 'admission_reason': 'admitted', 'logical_kv_blocks': 4, 'logical_kv_blocks_peak': 4, 'completion_tokens': 9, 'terminal_state': 'FINISHED', 'error_code': None, 'error_stage': None, 'queue_ms': 0.084647, 'ttft_ms': 1290.035263, 'tpot_ms': 0.07685462500000001, 'prefill_ms': 1290.035263, 'decode_ms': 0.6148370000000001, 'total_ms': 1304.768393, 'timeline': [{'phase': 'queue', 'start_ms': 13.997147, 'end_ms': 14.081794}, {'phase': 'prefill', 'start_ms': 14.118293, 'end_ms': 1304.153556}, {'phase': 'decode', 'start_ms': 1304.153556, 'end_ms': 1304.768393}, {'phase': 'request', 'start_ms': 0.0, 'end_ms': 1304.768393}]}
REAL_TORCH_GATEWAY=PASS
REAL_TORCH_PROMETHEUS=PASS


## 5. 真实 vLLM + CachePilot Gateway

vLLM 放在独立 Python 子进程中，避免 CUDA 动态库和 vLLM worker 污染 notebook 内核。该子进程会真实加载模型、生成 token、执行 abort，并通过真实 Gateway 检查 Trace 和 Prometheus。任何安装、CUDA、模型加载或请求失败都会直接失败，不降级、不 SKIP。

In [ ]:
import pathlib
import site
import subprocess
import textwrap

site_roots = {pathlib.Path(p) for p in site.getsitepackages()}
site_roots.add(pathlib.Path(site.getusersitepackages()))
library_dirs = set()
for site_root in site_roots:
    library_dirs.update(p for p in site_root.glob('nvidia/**/lib') if p.is_dir())
    torch_lib = site_root / 'torch' / 'lib'
    if torch_lib.is_dir():
        library_dirs.add(torch_lib)
library_dirs = sorted(str(p) for p in library_dirs)
smoke_env = os.environ.copy()
smoke_env['PYTHONPATH'] = str(WORKDIR) + os.pathsep + smoke_env.get('PYTHONPATH', '')
smoke_env['LD_LIBRARY_PATH'] = os.pathsep.join(
    library_dirs + ([smoke_env['LD_LIBRARY_PATH']] if smoke_env.get('LD_LIBRARY_PATH') else [])
)
smoke_env['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

vllm_script = textwrap.dedent(f'''
    import asyncio
    import inspect
    import pathlib
    from importlib import metadata
    from httpx import ASGITransport, AsyncClient

    from cachepilot.config.baseline import load_model_baseline
    from cachepilot.executors import VllmExecutor, VllmExecutorConfig
    from cachepilot.gateway.api import GatewaySettings, create_app
    from cachepilot.gateway.contracts import validate_chat_completion_request

    WORKDIR = pathlib.Path({str(WORKDIR)!r})
    LOCKED_VLLM = '0.24.0'

    def request(baseline, request_id, content, max_tokens, stream=True):
        return validate_chat_completion_request(
            {{
                'model': baseline.model_id,
                'messages': [{{'role': 'user', 'content': content}}],
                'stream': stream,
                'max_tokens': max_tokens,
            }},
            {{'X-Tenant-ID': 'team-a', 'X-Request-ID': request_id}},
            configured_model=baseline.model_id,
        )

    async def main():
        assert metadata.version('vllm') == LOCKED_VLLM
        baseline = load_model_baseline(WORKDIR / 'config/model.json')
        executor = VllmExecutor(
            VllmExecutorConfig(
                model_id=baseline.model_id,
                tokenizer_id=baseline.tokenizer_id,
                model_revision=baseline.model_revision,
                tokenizer_revision=baseline.tokenizer_revision,
                dtype='float16',
                context_limit=baseline.service_context_limit,
                gpu_memory_utilization=0.75,
                trust_remote_code=baseline.trust_remote_code,
                expected_vllm_version=LOCKED_VLLM,
            )
        )
        try:
            direct_request = request(baseline, 'real-vllm-direct', 'Reply with a short greeting.', 8)
            direct_items = [item async for item in executor.generate(direct_request)]
            direct_output = ''.join(item.text for item in direct_items)
            usage = executor.usage('real-vllm-direct')
            assert direct_output.strip()
            assert usage is not None
            assert usage.completion_tokens == sum(item.token_count for item in direct_items)
            assert usage.total_tokens == usage.prompt_tokens + usage.completion_tokens

            app = create_app(
                GatewaySettings(model_id=baseline.model_id),
                backend=executor,
                token_counter=executor,
            )
            async with AsyncClient(
                transport=ASGITransport(app=app),
                base_url='http://cachepilot',
            ) as client:
                response = await client.post(
                    '/v1/chat/completions',
                    headers={{'X-Tenant-ID': 'team-a', 'X-Request-ID': 'real-vllm-gateway'}},
                    json={{
                        'model': baseline.model_id,
                        'messages': [{{'role': 'user', 'content': 'Reply with a short greeting.'}}],
                        'stream': False,
                        'max_tokens': 8,
                    }},
                )
                assert response.status_code == 200, response.text
                assert response.json()['choices'][0]['message']['content'].strip()
                trace_response = await client.get(
                    '/v1/requests/real-vllm-gateway',
                    headers={{'X-Tenant-ID': 'team-a'}},
                )
                assert trace_response.status_code == 200
                trace = trace_response.json()['telemetry']
                assert trace['terminal_state'] == 'FINISHED'
                assert trace['prompt_tokens'] > 0
                assert trace['completion_tokens'] > 0
                assert trace['ttft_ms'] is not None
                metrics = (await client.get('/metrics')).text
                assert 'cachepilot_requests_total' in metrics
                assert 'cachepilot_ttft_seconds' in metrics
                for forbidden in ('request_id=', 'prompt=', 'prefix_key='):
                    assert forbidden not in metrics
                print('VLLM_GATEWAY_TELEMETRY=PASS')
                print('VLLM_PROMETHEUS_LOW_CARDINALITY=PASS')

            cancel_request = request(baseline, 'real-vllm-cancel', 'Write a long numbered list.', 64)
            stream = executor.generate(cancel_request)
            first_delta = await anext(stream)
            await executor.cancel('real-vllm-cancel', reason='explicit')
            remaining = [item async for item in stream]
            assert first_delta.text
            assert not remaining
            assert executor.cancel_reasons['real-vllm-cancel'] == 'explicit'
            print('VLLM_OUTPUT:', direct_output)
            print('VLLM_USAGE:', usage)
            print('VLLM_ABORT=PASS')
            print('VLLM_REAL_GPU=PASS')
        finally:
            shutdown = getattr(executor.engine, 'shutdown_background_loop', None)
            if callable(shutdown):
                result = shutdown()
                if inspect.isawaitable(result):
                    await result

    asyncio.run(main())
''')

vllm_run = subprocess.run(
    [sys.executable, '-c', vllm_script],
    cwd=WORKDIR,
    env=smoke_env,
    text=True,
    capture_output=True,
    timeout=1800,
)
print(redact_secrets(vllm_run.stdout[-12000:]))
if vllm_run.returncode != 0:
    print(redact_secrets(vllm_run.stderr[-12000:]))
    raise RuntimeError('Real vLLM GPU validation failed.')
assert 'VLLM_REAL_GPU=PASS' in vllm_run.stdout
assert 'VLLM_GATEWAY_TELEMETRY=PASS' in vllm_run.stdout
print('REAL_VLLM_GATEWAY=PASS')

In [ ]:
import subprocess
import sys
import time
from urllib.request import urlopen

GPU_SERVICE_URL = 'http://127.0.0.1:8000'
try:
    with urlopen(GPU_SERVICE_URL + '/readyz', timeout=2) as response:
        print('GPU service already running:', response.status)
except OSError:
    gpu_service = subprocess.Popen([
        sys.executable, 'main.py', '--host', '127.0.0.1', '--port', '8000',
    ], cwd=WORKDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
       text=True)
    deadline = time.monotonic() + 600
    while time.monotonic() < deadline:
        if gpu_service.poll() is not None:
            raise RuntimeError(f'GPU service exited with code {gpu_service.returncode}')
        try:
            with urlopen(GPU_SERVICE_URL + '/readyz', timeout=2) as response:
                if response.status == 200:
                    break
        except OSError:
            time.sleep(2)
    else:
        gpu_service.terminate()
        raise TimeoutError('GPU service did not become ready within 600 seconds')
    print('GPU service ready:', GPU_SERVICE_URL)


## 6. 渐进压测：上下文与并发边界

本节必须在真实 GPU runtime、TorchExecutor 服务就绪后执行。每个矩阵点重复三次，记录成功、429 过载保护、OOM、超时和失败；只有全部重复成功的点才会进入安全上限。单次成功不作为容量结论。

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

progressive_output = WORKDIR / 'runs' / 'colab-progressive-load'
progressive_command = [
    sys.executable, 'benchmarks/progressive_load.py',
    '--base-url', GPU_SERVICE_URL,
    '--output-dir', str(progressive_output),
    '--contexts', '32,128,512,2048,4096,8192',
    '--concurrencies', '1,2,4,8',
    '--repetitions', '3',
    '--max-tokens', '16',
    '--warmup',
]
progressive_run = subprocess.run(
    progressive_command, cwd=WORKDIR, text=True,
    capture_output=True, check=False,
)
print(progressive_run.stdout)
if progressive_run.stderr:
    print(progressive_run.stderr)
if progressive_run.returncode != 0:
    raise RuntimeError(f'progressive load failed with exit code {progressive_run.returncode}')
progressive_report = json.loads((progressive_output / 'progressive_report.json').read_text())
print(json.dumps(progressive_report['safety'], ensure_ascii=False, indent=2))
assert progressive_report['safety']['safe_context_tokens_observed'] is not None
print('PROGRESSIVE_LOAD=PASS')


## 6. 最终结论

只有以下结果全部出现，才算真实 GPU 验收通过：

- `GPU_PYTHON_PREFLIGHT=PASS`
- `GPU_RUNTIME=PASS`
- `REAL_TORCH_GATEWAY=PASS`
- `REAL_VLLM_GATEWAY=PASS`
- `VLLM_ABORT=PASS`
- `VLLM_GATEWAY_TELEMETRY=PASS`
- `VLLM_PROMETHEUS_LOW_CARDINALITY=PASS`

这些结果证明真实模型生成、Gateway 接入、取消、逐请求 Trace 和 Prometheus 指标都在当前 GPU 环境执行。它们仍不等于物理 Prefix 命中已可观测；如果执行器没有提供可验证的物理命中信号，该结论仍应显示为“不可观测”。

In [7]:
assert torch_output.strip()
assert vllm_run.returncode == 0
assert 'VLLM_REAL_GPU=PASS' in vllm_run.stdout
assert 'VLLM_GATEWAY_TELEMETRY=PASS' in vllm_run.stdout
print('REAL_GPU_ACCEPTANCE=PASS')
print('No deterministic backend or fake engine was used by this notebook.')

REAL_GPU_ACCEPTANCE=PASS
No deterministic backend or fake engine was used by this notebook.
